# Prä-Evaluation des Basismodells M0

Dieses Notebook lädt das finale Basismodell `M0` und den eingefrorenen
Pakistan--Islamabad-Benchmark. Vor dem Unlearning werden Prompt-Ziel-Paare,
Token-Grenzen, Sequenz-NLL, First-Token-Ränge und die Modell-Utility geprüft.

Explorative Starter-Benchmarks und temporäre Screenings zur Auswahl einzelner
Kontrollfakten wurden aus dieser bereinigten Fassung entfernt. Der finale
Benchmark wird unverändert aus den JSON-Dateien geladen.


## 1. Abhängigkeiten installieren


In [ ]:
!pip -q install tiktoken


## 2. Google Drive einbinden


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Importe, Seed und Gerät


In [ ]:
import json
import math
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch-Version: {torch.__version__}")
print(f"Verwendetes Gerät: {DEVICE}")


## 4. Projekt- und Modellkonfiguration


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/machine_unlearning_experiment")
BENCHMARK_DIR = PROJECT_DIR / "benchmark"
RESULTS_DIR = PROJECT_DIR / "results"

FACTS_PATH = BENCHMARK_DIR / "facts_pakistan-islamabad_final.json"
EXPERIMENTS_PATH = BENCHMARK_DIR / "experiment_pakistan_capital_islamabad_final.json"

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/model_checkpoints/"
    "clean_baseline_v1/model_final_step_0046460.pth"
)
VAL_TOKENS_PATH = Path(
    "/content/drive/MyDrive/simplewiki_val_tokens_exact.pt"
)

MODEL_ID = "M0"
EXPERIMENT_ID = "pakistan_capital_islamabad_final_v1"

MODEL_CONFIG = {
    "vocab_size": 50_257,
    "context_length": 1_024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fakten:      {FACTS_PATH}")
print(f"Experiment:  {EXPERIMENTS_PATH}")
print(f"Checkpoint:  {CHECKPOINT_PATH}")
print(f"Validation:  {VAL_TOKENS_PATH}")


## 5. GPT-Modellarchitektur


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError('d_out muss durch num_heads teilbar sein.')
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = queries @ keys.transpose(2, 3)
        causal_mask = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask, -torch.inf)
        attention_weights = torch.softmax(
            attention_scores / math.sqrt(self.head_dim), dim=-1
        )
        attention_weights = self.dropout(attention_weights)
        context = (attention_weights @ values).transpose(1, 2)
        context = context.reshape(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized + self.shift


class GELU(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 0.5 * x * (
            1.0 + torch.tanh(
                math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
            GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim']),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            num_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'],
            qkv_bias=cfg['qkv_bias'],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])]
        )
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    def forward(self, in_idx: torch.Tensor) -> torch.Tensor:
        _, sequence_length = in_idx.shape
        if sequence_length > self.pos_emb.num_embeddings:
            raise ValueError(
                f'Sequenzlänge {sequence_length} überschreitet die Context Length '
                f'{self.pos_emb.num_embeddings}.'
            )
        token_embeddings = self.tok_emb(in_idx)
        position_ids = torch.arange(sequence_length, device=in_idx.device)
        position_embeddings = self.pos_emb(position_ids)
        x = token_embeddings + position_embeddings
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


## 6. Tokenizer laden


In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')
assert tokenizer.n_vocab == MODEL_CONFIG['vocab_size']
print(f'Tokenizer-Vokabular: {tokenizer.n_vocab:,}')


## 7. Basismodell-Checkpoint laden


In [ ]:
def load_checkpoint_file(checkpoint_path: Path, map_location='cpu') -> Any:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint nicht gefunden: {checkpoint_path}')
    try:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(checkpoint_path, map_location=map_location)


def looks_like_state_dict(candidate: Any) -> bool:
    return (
        isinstance(candidate, dict)
        and bool(candidate)
        and all(
            isinstance(key, str) and isinstance(value, torch.Tensor)
            for key, value in candidate.items()
        )
    )


def extract_model_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if looks_like_state_dict(checkpoint):
        print('Checkpoint enthält direkt ein State-Dict.')
        return checkpoint
    if not isinstance(checkpoint, dict):
        raise TypeError('Der Checkpoint ist weder ein State-Dict noch ein Dictionary.')
    possible_keys = (
        'model_state_dict', 'model_state', 'state_dict', 'model',
        'MODEL_STATE_DICT', 'model_state_dict_cpu'
    )
    for key in possible_keys:
        candidate = checkpoint.get(key)
        if looks_like_state_dict(candidate):
            print(f'Modellgewichte gefunden unter Schlüssel {key!r}.')
            return candidate
    raise KeyError(
        'Kein Modell-State-Dict gefunden. '
        f'Checkpoint-Schlüssel: {list(checkpoint.keys())}'
    )


def strip_known_prefixes(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    cleaned = dict(state_dict)
    for prefix in ('module.', '_orig_mod.', 'model.'):
        if cleaned and all(key.startswith(prefix) for key in cleaned):
            cleaned = {key[len(prefix):]: value for key, value in cleaned.items()}
            print(f'Präfix {prefix!r} entfernt.')
    return cleaned


checkpoint = load_checkpoint_file(CHECKPOINT_PATH, map_location='cpu')
print(f'Checkpoint-Typ: {type(checkpoint)}')
if isinstance(checkpoint, dict):
    print('Checkpoint-Schlüssel:')
    for key in checkpoint.keys():
        print(f'  - {key}')

state_dict = strip_known_prefixes(extract_model_state_dict(checkpoint))
MODEL = GPTModel(MODEL_CONFIG)
load_result = MODEL.load_state_dict(state_dict, strict=True)
MODEL.to(DEVICE)
MODEL.eval()

parameter_count = sum(parameter.numel() for parameter in MODEL.parameters())
trainable_parameter_count = sum(
    parameter.numel() for parameter in MODEL.parameters() if parameter.requires_grad
)

print('\nModell erfolgreich geladen.')
print(load_result)
print(f'Parameter insgesamt:    {parameter_count:,}')
print(f'Trainierbare Parameter: {trainable_parameter_count:,}')

if isinstance(checkpoint, dict):
    print('\nVerfügbare Checkpoint-Metadaten:')
    found = False
    for key in ('global_step', 'tokens_seen', 'epoch', 'train_loss', 'val_loss', 'training_state'):
        if key in checkpoint:
            print(f'  {key}: {checkpoint[key]}')
            found = True
    if not found:
        print('  Keine bekannten Metadaten gefunden.')

del checkpoint, state_dict
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 8. Finalen Benchmark laden und validieren


In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(
            f"JSON-Datei nicht gefunden: {path}\n"
            "Prüfe den konfigurierten Benchmark-Pfad."
        )
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def validate_benchmark(facts_document, experiments_document) -> None:
    if "facts" not in facts_document:
        raise KeyError("In facts.json fehlt der Schlüssel 'facts'.")
    if "experiments" not in experiments_document:
        raise KeyError("In experiments.json fehlt der Schlüssel 'experiments'.")

    fact_ids = [fact["fact_id"] for fact in facts_document["facts"]]
    if len(fact_ids) != len(set(fact_ids)):
        raise ValueError("facts.json enthält doppelte fact_id-Werte.")
    fact_map = {fact["fact_id"]: fact for fact in facts_document["facts"]}

    prompt_ids = []
    for fact in facts_document["facts"]:
        required = {"fact_id", "subject", "relation", "object", "prompts"}
        missing = required - fact.keys()
        if missing:
            raise KeyError(
                f"Bei Fakt {fact.get('fact_id')} fehlen Felder: {sorted(missing)}"
            )
        if not fact["prompts"]:
            raise ValueError(f"Fakt {fact['fact_id']} besitzt keine Prompts.")
        for prompt in fact["prompts"]:
            required_prompt = {"prompt_id", "text", "target", "prompt_type"}
            missing_prompt = required_prompt - prompt.keys()
            if missing_prompt:
                raise KeyError(
                    f"Bei Prompt {prompt.get('prompt_id')} fehlen Felder: "
                    f"{sorted(missing_prompt)}"
                )
            prompt_ids.append(prompt["prompt_id"])
    if len(prompt_ids) != len(set(prompt_ids)):
        raise ValueError("facts.json enthält doppelte prompt_id-Werte.")

    experiment_ids = [
        experiment["experiment_id"]
        for experiment in experiments_document["experiments"]
    ]
    if len(experiment_ids) != len(set(experiment_ids)):
        raise ValueError("experiments.json enthält doppelte experiment_id-Werte.")

    for experiment in experiments_document["experiments"]:
        if "groups" not in experiment:
            raise KeyError(
                f"Experiment {experiment['experiment_id']} besitzt keine Gruppen."
            )
        assigned = []
        for group_name, group_fact_ids in experiment["groups"].items():
            for fact_id in group_fact_ids:
                if fact_id not in fact_map:
                    raise KeyError(
                        f"Unbekannte fact_id {fact_id!r} in Gruppe {group_name!r}."
                    )
                assigned.append(fact_id)
        duplicates = {
            fact_id for fact_id in assigned if assigned.count(fact_id) > 1
        }
        if duplicates:
            print(
                "Warnung: Fakten befinden sich in mehreren Gruppen: "
                f"{sorted(duplicates)}"
            )

    print(
        f"Benchmark gültig: {len(fact_ids)} Fakten, {len(prompt_ids)} Prompts "
        f"und {len(experiment_ids)} Experimente."
    )


facts_document = load_json(FACTS_PATH)
experiments_document = load_json(EXPERIMENTS_PATH)
validate_benchmark(facts_document, experiments_document)


## 9. Token-Grenzen prüfen

Prompt und Target werden getrennt tokenisiert. Zusätzlich wird geprüft, ob die
Tokenisierung der Konkatenation exakt dieser Aufteilung entspricht.


In [ ]:
def inspect_token_boundaries(facts_document, tokenizer) -> pd.DataFrame:
    rows = []

    for fact in facts_document["facts"]:
        for prompt in fact["prompts"]:
            targets = [("primary", prompt["target"])]
            targets.extend(
                ("alternative", target)
                for target in prompt.get("alternative_targets", [])
            )
            targets.extend(
                ("counterfactual", target)
                for target in prompt.get("counterfactual_targets", [])
            )

            for target_type, target in targets:
                prompt_ids = tokenizer.encode(prompt["text"])
                target_ids = tokenizer.encode(target)
                combined_ids = tokenizer.encode(prompt["text"] + target)

                rows.append({
                    "fact_id": fact["fact_id"],
                    "prompt_id": prompt["prompt_id"],
                    "target_type": target_type,
                    "prompt": prompt["text"],
                    "target": target,
                    "stable_boundary": combined_ids == prompt_ids + target_ids,
                    "prompt_token_count": len(prompt_ids),
                    "target_token_count": len(target_ids),
                })

    return pd.DataFrame(rows)


boundary_report = inspect_token_boundaries(
    facts_document,
    tokenizer,
)
invalid_boundaries = boundary_report[
    ~boundary_report["stable_boundary"]
]

if invalid_boundaries.empty:
    print("Alle Prompt-Ziel-Paare besitzen eine stabile Token-Grenze.")
else:
    print("Folgende Prompt-Ziel-Paare besitzen keine stabile Token-Grenze:")
    display(invalid_boundaries)


## 10. Sequenz-NLL und ergänzende Metriken


In [ ]:
@torch.no_grad()
def score_target_sequence(
    model,
    tokenizer,
    prompt: str,
    target: str,
) -> dict[str, Any]:
    model.eval()
    device = next(model.parameters()).device

    prompt_ids = tokenizer.encode(prompt)
    target_ids = tokenizer.encode(target)
    combined_ids = tokenizer.encode(prompt + target)

    if not prompt_ids:
        raise ValueError("Der Prompt darf nicht leer sein.")
    if not target_ids:
        raise ValueError("Das Target darf nicht leer sein.")
    if combined_ids != prompt_ids + target_ids:
        raise ValueError(
            "Prompt und Target besitzen keine stabile Token-Grenze.\n"
            f"Prompt: {prompt!r}\n"
            f"Target: {target!r}\n"
            "Bei GPT-2 sollte das Target meistens mit einem Leerzeichen beginnen."
        )

    input_length = len(combined_ids) - 1
    if input_length > MODEL_CONFIG["context_length"]:
        raise ValueError(
            f"Die Eingabe benötigt {input_length} Modellpositionen, "
            f"unterstützt werden {MODEL_CONFIG['context_length']}."
        )

    input_ids = torch.tensor(
        combined_ids[:-1],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    logits = model(input_ids)
    if logits.ndim != 3:
        raise ValueError(
            f"Erwartete Logits mit Form [B, T, V], erhalten: {tuple(logits.shape)}"
        )

    log_probs = F.log_softmax(logits, dim=-1)
    start_position = len(prompt_ids) - 1
    target_length = len(target_ids)
    relevant_log_probs = log_probs[
        0,
        start_position:start_position + target_length,
        :,
    ]

    if relevant_log_probs.shape[0] != target_length:
        raise RuntimeError("Nicht genügend Logits für alle Zieltoken vorhanden.")

    target_tensor = torch.tensor(
        target_ids,
        dtype=torch.long,
        device=device,
    )
    token_log_probs = relevant_log_probs.gather(
        dim=1,
        index=target_tensor.unsqueeze(1),
    ).squeeze(1)
    token_probabilities = token_log_probs.exp()

    sequence_nll = -token_log_probs.mean().item()
    total_sequence_nll = -token_log_probs.sum().item()
    geo_mean_probability = math.exp(-sequence_nll)

    first_token_logits = logits[0, start_position]
    first_target_token_id = target_ids[0]
    first_target_logit = first_token_logits[first_target_token_id]
    first_token_rank = (
        int((first_token_logits > first_target_logit).sum().item()) + 1
    )
    first_token_probability = torch.softmax(
        first_token_logits,
        dim=-1,
    )[first_target_token_id].item()

    return {
        "sequence_nll": sequence_nll,
        "total_sequence_nll": total_sequence_nll,
        "geo_mean_probability": geo_mean_probability,
        "first_token_probability": first_token_probability,
        "first_token_rank": first_token_rank,
        "target_token_count": target_length,
        "target_token_ids": target_ids,
        "target_tokens": [
            tokenizer.decode([token_id]) for token_id in target_ids
        ],
        "token_log_probs": token_log_probs.cpu().tolist(),
        "token_probabilities": token_probabilities.cpu().tolist(),
    }


## 11. Prä-Evaluation ausführen


In [ ]:
def get_experiment(experiments_document, experiment_id: str) -> dict[str, Any]:
    for experiment in experiments_document['experiments']:
        if experiment['experiment_id'] == experiment_id:
            return experiment
    available = [e['experiment_id'] for e in experiments_document['experiments']]
    raise KeyError(f'Experiment {experiment_id!r} nicht gefunden. Verfügbar: {available}')


def run_pre_evaluation(
    model,
    tokenizer,
    facts_document,
    experiments_document,
    experiment_id: str,
    model_id: str,
    checkpoint_path: Path,
) -> pd.DataFrame:
    fact_map = {fact['fact_id']: fact for fact in facts_document['facts']}
    experiment = get_experiment(experiments_document, experiment_id)
    rows = []

    for group_name, fact_ids in experiment['groups'].items():
        for fact_id in fact_ids:
            fact = fact_map[fact_id]
            for prompt in fact['prompts']:
                targets = [('primary', prompt['target'])]

                targets.extend(
                    ('alternative', alternative)
                    for alternative in prompt.get('alternative_targets', [])
                )

                targets.extend(
                    ("counterfactual", target)
                    for target in prompt.get(
                        "counterfactual_targets",
                        [],
                    )
                )

                for target_type, target in targets:
                    scores = score_target_sequence(
                        model=model,
                        tokenizer=tokenizer,
                        prompt=prompt['text'],
                        target=target,
                    )
                    rows.append({
                        'model_id': model_id,
                        'checkpoint': checkpoint_path.name,
                        'experiment_id': experiment_id,
                        'group': group_name,
                        'fact_id': fact_id,
                        'subject': fact['subject'],
                        'relation': fact['relation'],
                        'object': fact['object'],
                        'prompt_id': prompt['prompt_id'],
                        'prompt_type': prompt['prompt_type'],
                        'orientation': prompt.get('orientation'),
                        'prompt': prompt['text'],
                        'target': target,
                        'target_type': target_type,
                        **scores,
                    })

    if not rows:
        raise RuntimeError('Das Experiment enthielt keine auswertbaren Prompt-Ziel-Paare.')
    return pd.DataFrame(rows)


RESULTS = run_pre_evaluation(
    model=MODEL,
    tokenizer=tokenizer,
    facts_document=facts_document,
    experiments_document=experiments_document,
    experiment_id=EXPERIMENT_ID,
    model_id=MODEL_ID,
    checkpoint_path=CHECKPOINT_PATH,
)
print(f'{len(RESULTS)} Prompt-Ziel-Kombinationen ausgewertet.')
display(RESULTS.head())


## 12. Primäre Zielantworten und Aggregationen


In [ ]:
PRIMARY_RESULTS = RESULTS[
    RESULTS["target_type"] == "primary"
].copy()


def derive_analysis_group(row):
    """
    Trennt nur bei Ziel- und Same-Relation-Fakten
    zwischen direkter und inverser Richtung.

    Alle übrigen Gruppen bleiben unverändert.
    """
    group = row["group"]

    if group not in {"target", "same_relation"}:
        return group

    # Bevorzugt explizite orientation aus facts.json,
    # sofern sie in RESULTS übernommen wurde.
    orientation = row.get("orientation", None)

    if orientation == "country_to_capital":
        return f"{group}_direct"

    if orientation == "capital_to_country":
        return f"{group}_inverse"

    # Fallback über prompt_type
    prompt_type = str(
        row.get("prompt_type", "")
    ).lower()

    if "inverse" in prompt_type:
        return f"{group}_inverse"

    if (
        "direct" in prompt_type
        or "possessive" in prompt_type
    ):
        return f"{group}_direct"

    raise ValueError(
        "Prompt-Richtung konnte nicht bestimmt werden: "
        f"group={group!r}, "
        f"prompt_type={prompt_type!r}, "
        f"orientation={orientation!r}"
    )


PRIMARY_RESULTS["analysis_group"] = (
    PRIMARY_RESULTS.apply(
        derive_analysis_group,
        axis=1,
    )
)


display(
    PRIMARY_RESULTS[
        [
            "group",
            "analysis_group",
            "fact_id",
            "prompt_id",
            "prompt",
            "target",
            "sequence_nll",
            "geo_mean_probability",
            "first_token_probability",
            "first_token_rank",
            "target_token_count",
        ]
    ].sort_values(
        [
            "analysis_group",
            "fact_id",
            "sequence_nll",
        ]
    )
)


In [ ]:
FACT_SUMMARY = (
    PRIMARY_RESULTS
    .groupby(
        [
            "group",
            "analysis_group",
            "fact_id",
            "subject",
            "relation",
            "object",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        mean_sequence_nll=(
            "sequence_nll",
            "mean",
        ),
        std_sequence_nll=(
            "sequence_nll",
            "std",
        ),
        mean_geo_probability=(
            "geo_mean_probability",
            "mean",
        ),
        median_first_token_rank=(
            "first_token_rank",
            "median",
        ),
        max_first_token_rank=(
            "first_token_rank",
            "max",
        ),
        prompt_count=(
            "prompt_id",
            "nunique",
        ),
        target_token_count=(
            "target_token_count",
            "max",
        ),
    )
)

FACT_SUMMARY["std_sequence_nll"] = (
    FACT_SUMMARY[
        "std_sequence_nll"
    ].fillna(0.0)
)


display(
    FACT_SUMMARY.sort_values(
        [
            "analysis_group",
            "mean_sequence_nll",
        ]
    )
)


In [ ]:
GROUP_SUMMARY = (
    FACT_SUMMARY
    .groupby(
        "analysis_group",
        as_index=False,
    )
    .agg(
        fact_count=(
            "fact_id",
            "nunique",
        ),
        prompt_count=(
            "prompt_count",
            "sum",
        ),
        mean_fact_nll=(
            "mean_sequence_nll",
            "mean",
        ),
        std_fact_nll=(
            "mean_sequence_nll",
            "std",
        ),
        median_fact_rank=(
            "median_first_token_rank",
            "median",
        ),
        mean_fact_geo_probability=(
            "mean_geo_probability",
            "mean",
        ),
    )
)

GROUP_SUMMARY["std_fact_nll"] = (
    GROUP_SUMMARY[
        "std_fact_nll"
    ].fillna(0.0)
)


display(
    GROUP_SUMMARY.sort_values(
        "analysis_group"
    )
)


## 13. Alternative und Counterfactual-Targets


In [ ]:
ALTERNATIVE_RESULTS = RESULTS[
    RESULTS["target_type"] == "alternative"
].copy()

COUNTERFACTUAL_RESULTS = RESULTS[
    RESULTS["target_type"] == "counterfactual"
].copy()

DISPLAY_COLUMNS = [
    "group",
    "fact_id",
    "prompt_id",
    "prompt",
    "target",
    "target_type",
    "sequence_nll",
    "geo_mean_probability",
    "first_token_probability",
    "first_token_rank",
    "target_token_count",
]

if ALTERNATIVE_RESULTS.empty:
    print("Keine alternativen Zielantworten im finalen Benchmark.")
else:
    display(
        ALTERNATIVE_RESULTS[DISPLAY_COLUMNS].sort_values(
            ["group", "fact_id", "sequence_nll"]
        )
    )

if COUNTERFACTUAL_RESULTS.empty:
    print("Keine Counterfactual-Targets im finalen Benchmark.")
else:
    display(
        COUNTERFACTUAL_RESULTS[DISPLAY_COLUMNS].sort_values(
            ["fact_id", "prompt_id", "sequence_nll"]
        )
    )


## 14. Modell-Utility auf dem festen Validierungsausschnitt

Für die Prä-Evaluation wird derselbe feste Ausschnitt von 50
Validierungsbatches verwendet wie später während der Unlearning-Läufe.


In [ ]:
class CachedGPTDataset(Dataset):
    def __init__(
        self,
        token_tensor: torch.Tensor,
        max_length: int,
        stride: int,
    ) -> None:
        if token_tensor.ndim != 1:
            raise ValueError(
                "token_tensor muss eindimensional sein."
            )
        if max_length <= 0:
            raise ValueError(
                "max_length muss positiv sein."
            )
        if stride <= 0:
            raise ValueError(
                "stride muss positiv sein."
            )
        if len(token_tensor) <= max_length:
            raise ValueError(
                "Der Token-Tensor ist zu kurz für "
                "ein vollständiges Trainingsfenster."
            )

        self.token_tensor = token_tensor
        self.max_length = max_length
        self.stride = stride

        self.num_windows = 1 + (
            len(token_tensor)
            - max_length
            - 1
        ) // stride

    def __len__(self) -> int:
        return self.num_windows

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if (
            index < 0
            or index >= self.num_windows
        ):
            raise IndexError(index)

        start = index * self.stride

        input_ids = self.token_tensor[
            start:
            start + self.max_length
        ]

        target_ids = self.token_tensor[
            start + 1:
            start + self.max_length + 1
        ]

        return input_ids, target_ids


VAL_TOKENS_PATH = Path(
    "/content/drive/MyDrive/simplewiki_val_tokens_exact.pt"
)

val_data = torch.load(
    VAL_TOKENS_PATH,
    map_location="cpu",
)

VAL_TOKENS = val_data["val_tokens"]

print(
    f"Validation-Tokens geladen: "
    f"{len(VAL_TOKENS):,}"
)

print(
    f"Originaler Split-Index: "
    f"{val_data['split_index']:,}"
)


# ---------------------------------------------------------
# Validation-Dataset
# ---------------------------------------------------------

VALIDATION_CONTEXT_LENGTH = 1024
VALIDATION_STRIDE = 1024
VALIDATION_BATCH_SIZE = 4

VAL_DATASET = CachedGPTDataset(
    token_tensor=VAL_TOKENS,
    max_length=VALIDATION_CONTEXT_LENGTH,
    stride=VALIDATION_STRIDE,
)

print(
    f"Validation Windows: "
    f"{len(VAL_DATASET):,}"
)


# ---------------------------------------------------------
# Validation-Loader
# ---------------------------------------------------------

VAL_LOADER = DataLoader(
    VAL_DATASET,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

print(
    f"Validation Batches: "
    f"{len(VAL_LOADER):,}"
)

assert val_data["context_length"] == VALIDATION_CONTEXT_LENGTH
assert val_data["stride"] == VALIDATION_STRIDE

if "batch_size" in val_data:
    assert val_data["batch_size"] == VALIDATION_BATCH_SIZE

print("Validation-Konfiguration erfolgreich geprüft.")

def calc_loss_batch(
    input_batch: torch.Tensor,
    target_batch: torch.Tensor,
    model: nn.Module,
    device: torch.device,
) -> torch.Tensor:
    input_batch = input_batch.to(
        device,
        non_blocking=True,
    )

    target_batch = target_batch.to(
        device,
        non_blocking=True,
    )

    logits = model(input_batch)

    return F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
    )

@torch.no_grad()
def calc_loss_loader(
    data_loader: DataLoader,
    model: nn.Module,
    device: torch.device,
    num_batches: int | None = None,
) -> float:
    if len(data_loader) == 0:
        return float("nan")

    batches_to_evaluate = (
        len(data_loader)
        if num_batches is None
        else min(
            num_batches,
            len(data_loader),
        )
    )

    total_loss = 0.0

    for batch_index, (
        input_batch,
        target_batch,
    ) in enumerate(data_loader):

        if batch_index >= batches_to_evaluate:
            break

        loss = calc_loss_batch(
            input_batch,
            target_batch,
            model,
            device,
        )

        total_loss += loss.item()

    return (
        total_loss
        / batches_to_evaluate
    )
@torch.no_grad()
def evaluate_validation_loss(
    model: nn.Module,
    val_loader: DataLoader,
    device: torch.device,
    eval_batches: int,
) -> float:
    was_training = model.training
    model.eval()

    val_loss = calc_loss_loader(
        data_loader=val_loader,
        model=model,
        device=device,
        num_batches=eval_batches,
    )

    model.train(was_training)

    return val_loss

VALIDATION_EVAL_BATCHES = 50

M0_VALIDATION_LOSS = (
    evaluate_validation_loss(
        model=MODEL,
        val_loader=VAL_LOADER,
        device=DEVICE,
        eval_batches=(
            VALIDATION_EVAL_BATCHES
        ),
    )
)

print(
    f"M0 Validation Loss "
    f"({VALIDATION_EVAL_BATCHES} Batches): "
    f"{M0_VALIDATION_LOSS:.6f}"
)


## 15. Ergebnisse und Metadaten speichern


In [ ]:
run_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

run_name = (
    f"{MODEL_ID}_"
    f"{EXPERIMENT_ID}_"
    f"{run_timestamp}"
)

run_directory = RESULTS_DIR / run_name
run_directory.mkdir(
    parents=True,
    exist_ok=False,
)


# ---------------------------------------------------------
# Vollständige Prompt-Ergebnisse
# ---------------------------------------------------------

RESULTS.to_csv(
    run_directory / "prompt_results.csv",
    index=False,
)

RESULTS.to_json(
    run_directory / "prompt_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)


# ---------------------------------------------------------
# Nur primäre Zielantworten
# ---------------------------------------------------------

PRIMARY_RESULTS.to_csv(
    run_directory
    / "primary_prompt_results.csv",
    index=False,
)


# ---------------------------------------------------------
# Aggregationen
# ---------------------------------------------------------

FACT_SUMMARY.to_csv(
    run_directory / "fact_summary.csv",
    index=False,
)

GROUP_SUMMARY.to_csv(
    run_directory / "group_summary.csv",
    index=False,
)


# ---------------------------------------------------------
# Alternative Zielantworten
# ---------------------------------------------------------

if not ALTERNATIVE_RESULTS.empty:
    ALTERNATIVE_RESULTS.to_csv(
        run_directory
        / "alternative_target_results.csv",
        index=False,
    )

if not COUNTERFACTUAL_RESULTS.empty:
    COUNTERFACTUAL_RESULTS.to_csv(
        run_directory
        / "counterfactual_target_results.csv",
        index=False,
    )

# ---------------------------------------------------------
# Utility-Evaluation
# ---------------------------------------------------------

UTILITY_SUMMARY = pd.DataFrame([
    {
        "model_id": MODEL_ID,
        "checkpoint": CHECKPOINT_PATH.name,
        "validation_loss": float(
            M0_VALIDATION_LOSS
        ),
        "validation_batches": (
            VALIDATION_EVAL_BATCHES
        ),
        "validation_context_length": (
            VALIDATION_CONTEXT_LENGTH
        ),
        "validation_stride": (
            VALIDATION_STRIDE
        ),
        "validation_batch_size": (
            VALIDATION_BATCH_SIZE
        ),
    }
])

UTILITY_SUMMARY.to_csv(
    run_directory / "utility_summary.csv",
    index=False,
)


# ---------------------------------------------------------
# Metadaten
# ---------------------------------------------------------

run_metadata = {
    "run_name": run_name,
    "model_id": MODEL_ID,
    "experiment_id": EXPERIMENT_ID,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "device": str(DEVICE),
    "seed": SEED,
    "timestamp_utc": run_timestamp,

    "model_config": MODEL_CONFIG,
    "parameter_count": parameter_count,

    "prompt_target_evaluations": int(
        len(RESULTS)
    ),
    "primary_prompt_evaluations": int(
        len(PRIMARY_RESULTS)
    ),
    "evaluated_facts": int(
        PRIMARY_RESULTS[
            "fact_id"
        ].nunique()
    ),

    "analysis_groups": sorted(
        PRIMARY_RESULTS[
            "analysis_group"
        ].dropna().unique().tolist()
    ),

    "benchmark_version": (
        facts_document.get(
            "benchmark_version"
        )
    ),
   "validation_loss": float(
    M0_VALIDATION_LOSS
    ),
    "validation_batches": (
        VALIDATION_EVAL_BATCHES
    ),
    "validation_context_length": (
        VALIDATION_CONTEXT_LENGTH
    ),
    "validation_stride": (
        VALIDATION_STRIDE
    ),
    "validation_batch_size": (
        VALIDATION_BATCH_SIZE
    ),

    "analysis_schema_version": "1.0",
}


# ---------------------------------------------------------
# Snapshots
# ---------------------------------------------------------

for path, document in (
    (
        run_directory / "metadata.json",
        run_metadata,
    ),
    (
        run_directory / "facts_snapshot.json",
        facts_document,
    ),
    (
        run_directory
        / "experiments_snapshot.json",
        experiments_document,
    ),
):
    with path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            document,
            file,
            ensure_ascii=False,
            indent=2,
        )


print(
    "Ergebnisse gespeichert unter:\n"
    f"{run_directory}"
)


## 16. Verwendung im Hauptexperiment

Die hier geladenen Benchmarkdateien bleiben für alle Unlearning-Läufe
unverändert. Ein positives
\(\Delta\mathrm{NLL}=\mathrm{NLL}_{neu}-\mathrm{NLL}_{M0}\)
bedeutet später, dass die jeweilige Zielsequenz gegenüber `M0`
unwahrscheinlicher geworden ist.
